# YouTube data quality check

**Input:** `data/interim/youtube.csv` — monthly YouTube Content ID reports, 01-2024 → 09-2026.

**Goal:** find data problems *before* cleaning. This notebook only looks at the data and changes nothing; the fixes go to `src/clean_data.py`.

A summary table is at the end.

In [24]:
import pandas as pd

data = pd.read_csv("../data/interim/youtube.csv", parse_dates=["report_month"])

## 1. Overview

**Check:** table size and first rows.

In [ ]:
# Size of the table (rows, columns) and a first look at the data
print(data.shape)
data.head()

(34470, 6)


**Found:** 34,470 rows, 6 columns. The period and the number of months are checked in section 9.

**Decision:** matches the parser output.

## 2. Column types

In [29]:
data.dtypes

artist                     str
track                      str
revenue                float64
video_url                  str
payment_type               str
report_month    datetime64[us]
dtype: object

**Found:** `revenue` is a float, `report_month` is a datetime, the other columns are text.

**Decision:** no action needed.

## 3. Missing values

**Check:** are there empty cells?

In [30]:
data.isna().sum()

artist          0
track           0
revenue         0
video_url       0
payment_type    0
report_month    0
dtype: int64

**Found:** no missing values in any column.

**Decision:** no action needed.

## 4. Negative and zero revenue

**Check:** royalties should not be negative; zero-revenue rows add nothing to the analysis.

In [31]:
print("Negative revenue:", (data["revenue"] < 0).sum())
print("Zero revenue:", (data["revenue"] == 0).sum())

Negative revenue: 0
Zero revenue: 0


**Found:** no negative and no zero values.

**Decision:** no action needed.

## 5. Duplicate rows

In [32]:
data.duplicated().sum()

np.int64(0)

**Found:** 0 identical rows.

**Decision:** no action needed.

## 6. Text columns: artist, track, payment type

**Check:** each column has a small set of values, and the same name is not written in different ways (extra spaces, upper/lower case).

In [ ]:
data["artist"].value_counts()

In [ ]:
data["track"].value_counts()

In [ ]:
data["payment_type"].value_counts()

In [35]:
# Same name written differently? Compare the number of unique values
# before and after removing spaces and switching to lower case
for col in ["artist", "track", "payment_type"]:
    raw = data[col].nunique()
    normalized = data[col].str.strip().str.lower().nunique()
    print(f"{col}: {raw} unique, {normalized} after strip + lower")

artist: 6 unique, 6 after strip + lower
track: 18 unique, 18 after strip + lower
payment_type: 2 unique, 2 after strip + lower


**Found:**
- 6 artists, 18 tracks, 2 payment types: `ads` (advertising) and `red` (YouTube Premium).
- The number of unique values does not change after strip + lower → no spelling variants.

**Decision:** no action needed. There is no ISRC, so in the analysis a track is always identified by the pair `artist` + `track`.

## 7. Video URL format

**Check:** every `video_url` is a standard YouTube link: `https://www.youtube.com/watch?v=` + an 11-character video ID.

In [ ]:
# Standard link = fixed prefix + 11 characters (letters, digits, "_" or "-")
pattern = r"^https://www\.youtube\.com/watch\?v=[A-Za-z0-9_-]{11}$"
is_standard = data["video_url"].str.match(pattern)
print("Non-standard URLs:", (~is_standard).sum())
data[~is_standard]

**Found:** 2 rows (one month, one track, tiny revenue) have a non-standard link of the form `watch?v=deleted-...` — the video was deleted, so YouTube replaced its ID.

**Decision:** keep both rows — the revenue was really paid to the artist. In `clean_data.py` replace their `video_url` with `deleted`. `video_url` is kept only as a reference to open a video when needed; links themselves are not analysed.

## 8. Largest revenue values

**Check:** are the biggest amounts plausible, or do they look like typos?

In [ ]:
data["revenue"].describe()

In [ ]:
# The 10 rows with the highest revenue: which video, month and payment type
data.nlargest(10, "revenue")

**Found:**
- Revenue per row is extremely skewed: most rows earn a tiny amount (the median is far below one cent), while the mean is hundreds of times larger than the median and the maximum is far above both. A few very large rows pull the mean up.
- All 10 largest rows are the **same video** (one track, `ads`) in different months, with stable amounts month to month — a steady earner, not a one-off typo.

**Decision:** no action needed — all values, including the largest ones, are plausible (confirmed by the owner). The concentration of revenue in a few videos is a topic for the analysis notebooks, not a cleaning issue.

## 9. Rows and revenue per month

**Check:** every month between the first and last report is present, and the number of rows does not jump unexpectedly.

In [ ]:
# Rows and total revenue in each monthly report
monthly = data.groupby("report_month").agg(
    rows=("revenue", "size"),
    revenue=("revenue", "sum"),
)
monthly

In [42]:
# Period covered, and is any month missing between the first and the last report?
all_months = pd.date_range(data["report_month"].min(), data["report_month"].max(), freq="MS")
missing = all_months.difference(monthly.index)
print("Period:", all_months[0].date(), "->", all_months[-1].date(), f"({len(all_months)} months)")
print("Missing months:", list(missing))

Period: 2024-01-01 -> 2026-09-01 (33 months)
Missing months: []


**Found:**
- All 33 months (01-2024 → 09-2026) are present, no gaps.
- Revenue falls noticeably from 04/05-2025. This is not a data error: the decline is analysed in a separate notebook.
- From 06-2026 there are only **6–7 rows per month**, and **09-2026** has almost zero revenue.

**Decision:**
- No gaps → no action needed.
- 2025 decline → real business dynamics, kept as is and analysed separately.
- 06–09 2026 → real data, kept as is. The contract between the label and the distributor was terminated, and the catalogue was gradually taken down to be re-released elsewhere. In the analysis these months are treated as a **wind-down period**: they are marked on charts and not compared directly with earlier months.

## 10. Control total

**Check:** total revenue matches the sum of the source Excel reports.

In [ ]:
round(data["revenue"].sum(), 2)

**Found:** the total matches the sum of the source Excel reports, computed separately.

**Decision:** no rows were lost or duplicated by the parser.

## Summary

| # | Check | Found | Decision |
|---|---|---|---|
| 1 | Overview | 34,470 rows, 6 columns | OK |
| 2 | Column types | as expected | no action |
| 3 | Missing values | none | no action |
| 4 | Negative / zero revenue | none | no action |
| 5 | Duplicate rows | none | no action |
| 6 | Artist / track / payment type | 6 / 18 / 2 values, no spelling variants | no action |
| 7 | Video URL format | 2 deleted-video links | keep rows, set `video_url` to `deleted` |
| 8 | Largest values | strongly skewed revenue, all values plausible | no action |
| 9 | Rows per month | 33 months, no gaps; 06–09 2026 catalogue wind-down | keep; treat 06–09 2026 as wind-down period |
| 10 | Control total | matches | OK |

**Next step:** `02_dsp_data_quality.ipynb`.